# Validate formation and position mapping

This notebook applies the reusable position mapping in `src/positions/formation_mapping.py` to the eligible Nottingham Forest defender appearances. It keeps the provider decoding separate from the project's standardised scouting positions.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (root for root in search_roots if (root / 'src').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.positions import add_position_mapping

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
PROJECT_ROOT

## Load eligible appearances

The compact processed table is sufficient for this validation because it already retains `formation_code` and `lineup_slot`.

In [ ]:
APPEARANCE_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'processed'
    / 'forest_eligible_defender_appearances.parquet'
)

eligible_defender_appearances = pd.read_parquet(APPEARANCE_PATH)
mapped_defender_appearances = add_position_mapping(
    eligible_defender_appearances
)

print(f'Eligible appearances: {len(mapped_defender_appearances):,}')
mapped_defender_appearances.head()

## Mapping coverage

Every unresolved combination should remain visible rather than being guessed.

In [ ]:
mapping_coverage = (
    mapped_defender_appearances
    .groupby(
        ['mapping_status', 'mapping_confidence'],
        dropna=False,
    )
    .size()
    .rename('appearance_count')
    .reset_index()
    .sort_values('appearance_count', ascending=False)
)
mapping_coverage

In [ ]:
formation_slot_summary = (
    mapped_defender_appearances
    .groupby(
        [
            'formation_code',
            'formation_name',
            'lineup_slot',
            'decoded_provider_position',
            'standardised_position',
            'position_group',
            'mapping_status',
        ],
        dropna=False,
    )
    .size()
    .rename('appearance_count')
    .reset_index()
    .sort_values(['formation_code', 'lineup_slot'])
)
formation_slot_summary

## Inspect mapped appearances

In [ ]:
appearance_columns = [
    'match_date',
    'match_description',
    'player_name',
    'formation_code',
    'formation_name',
    'lineup_slot',
    'decoded_provider_position',
    'standardised_position',
    'position_group',
    'mapping_confidence',
    'mapping_status',
    'minutes_played',
]

(
    mapped_defender_appearances[appearance_columns]
    .sort_values(['match_date', 'lineup_slot'])
    .reset_index(drop=True)
)

## Review unresolved appearances

These rows are excluded from position templates until a defensible rule is available.

In [ ]:
unresolved_appearances = mapped_defender_appearances.loc[
    mapped_defender_appearances['mapping_status'].ne('mapped'),
    appearance_columns,
].sort_values(['mapping_status', 'match_date', 'player_name'])

unresolved_appearances

## High-confidence centre-back sample

This is the appearance population that would feed the first centre-back template.

In [ ]:
eligible_centre_back_appearances = mapped_defender_appearances.loc[
    mapped_defender_appearances['position_group'].eq('Centre-back')
    & mapped_defender_appearances['mapping_confidence'].eq('high')
].copy()

print(
    f'High-confidence centre-back appearances: '
    f'{len(eligible_centre_back_appearances):,}'
)
eligible_centre_back_appearances[appearance_columns].head(20)